# 09_加载比赛基线

**学习目标：** 加载已有本地微调权重，检查40分数，并生成4段流程预测。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

本课使用现有initial16-20261020-5ep模型，方便与基础R(2+1)D34对应。它不是本次新训练的模型，也不是现有项目最复杂的优化方案。模型初始化只加载本地参数；输入预处理必须与训练一致。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 分数、类别编号和动作名


In [ ]:
import numpy as np
model, run = L.load_reference_model()
path, row, classes = L.sample_path('validation', 0)
scores = np.asarray(model.predict(path))
assert scores.shape == (40,) and np.isfinite(scores).all()
predicted = int(scores.argmax())
print('本地权重目录', run.name)
print('分数形状', scores.shape, '预测', predicted, classes[predicted], '真实', classes[row['label']])
print('返回logits，不要求40分数之和为1。')


## 释放当前模型，独立生成4段预测


In [ ]:
import gc
import torch
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
result = L.predict(limit=4)
print('样本数', result['samples'], '完整验证？', result['full_validation'])
print('新CSV：', result['csv'])
for item in result['details']:
    print(item['index'], item['truth'], '→', item['prediction'])


## 动手练习

从scores找分数最高的三个类别编号，按分数从高到低排列。填入your_answer。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：np.argsort后倒序取3项
if your_answer is None:
    print('待完成：不要把top3的分数当作类别编号')
else:
    assert list(your_answer) == np.argsort(scores)[::-1][:3].tolist()
    print('练习通过')


## 常见误区

4段抽样不是完整466段评估；旧模型的成绩不能归为自己的新实验。


## 过关检查与选读

能从新终端复现预测；解释Model/config/weights_path；理解score、class_id、class_name。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/09_加载比赛基线.ipynb) · [官方选读](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html)

完成后在README.md#学习记录写下一句自己的解释。
